In [4]:
import numpy as np
np.bool = np.bool_ 
import pandas as pd
import os
from tqdm import tqdm

import mxnet as mx
from mxnet import gluon
from mxnet import autograd
from mxnet import image

import sys
sys.path.append('resuneta/src')
sys.path.append('decode/FracTAL_ResUNet/models/semanticsegmentation')
sys.path.append('decode/FracTAL_ResUNet/nn/loss')
sys.path.append('autogeobound/MXNet-ResUNeta/')

from bound_dist import get_distance, get_boundary
from FracTAL_ResUNet import FracTAL_ResUNet_cmtsk
#from ftnmt_loss import ftnmt_loss
from datasets import *

from sklearn.metrics import matthews_corrcoef

import random

import rasterio
from rasterio.enums import ColorInterp
from rasterio.features import shapes
from scipy import ndimage

import matplotlib.pyplot as plt
%matplotlib inline

import higra as hg
import torch
import pickle

import itertools

import imageio.v3 as imageio
from skimage.color import label2rgb
import cv2
from skimage.segmentation import mark_boundaries
from skimage.segmentation import find_boundaries

from shapely.geometry import shape
import shapely
from shapely import Polygon, orient_polygons

import ast

import glob

In [5]:
import random
np.random.seed(0)
random.seed(0)

image_filenames = sorted(glob.glob(os.path.join('inference_data/images', "*.tif")))
image_filenames = random.sample(image_filenames, 100)

# First attempt

In [6]:
n_filters = 32
depth = 6
n_classes = 1
ctx = mx.gpu(2)
model = FracTAL_ResUNet_cmtsk(nfilters_init=n_filters, depth=depth, NClasses=n_classes)
model.load_parameters('experiments/8m-resolution/US/fractal-resunet_NAIP_nfilter-32_depth-6_bs-80_lr-0.001/model-20epoch.params', ctx=ctx)

def predict_boundaries_512(image):
    bound = np.zeros((512, 512))
    count = np.zeros((512, 512))
    topleft_indices = list(itertools.product([0, 128, 256], [0, 128, 256]))
    for topleft_index in topleft_indices:
        x1, x2, y1, y2 = topleft_index[0], topleft_index[0]+256, topleft_index[1], topleft_index[1]+256
        image_crop = mx.nd.array(image[:, :, x1:x2, y1:y2])
        _, bound_crop, _ = model(image_crop.as_in_context(ctx))
        bound[x1:x2, y1:y2] += bound_crop[0][0].asnumpy()
        count[x1:x2, y1:y2] += np.ones((256, 256))
    bound /= count
    return bound

depth:= 0, nfilters: 32, nheads::8, widths::1
depth:= 1, nfilters: 64, nheads::16, widths::1
depth:= 2, nfilters: 128, nheads::32, widths::1
depth:= 3, nfilters: 256, nheads::64, widths::1
depth:= 4, nfilters: 512, nheads::128, widths::1
depth:= 5, nfilters: 1024, nheads::256, widths::1
depth:= 6, nfilters: 512, nheads::256, widths::1
depth:= 7, nfilters: 256, nheads::128, widths::1
depth:= 8, nfilters: 128, nheads::64, widths::1
depth:= 9, nfilters: 64, nheads::32, widths::1
depth:= 10, nfilters: 32, nheads::16, widths::1


In [12]:
T_values = [0.003, 0.005, 0.02, 0.04, 0.09, 0.19]
graph = hg.get_8_adjacency_graph((512, 512))
topleft_indices = list(itertools.product([0, 512, 1024, 1536], [0, 512, 1024, 1536]))

image_border = np.ones((2048, 2048))
image_border[1:-1, 1:-1] = 0

image_bottom_right_corner = np.zeros((2048, 2048))
image_bottom_right_corner[512:, 512:] = 1

for image_filename in tqdm(image_filenames):
    image_number = image_filename[-9:-4]
    
    src = rasterio.open(image_filename)
    transform = src.transform
    
    image = imageio.imread(image_filename)
    image = np.moveaxis(image, -1, 0)[np.newaxis, :]

    CDL_filename = image_filename.replace("images", "CDL")
    CDL = imageio.imread(CDL_filename)

    
    trees_list = []
    altitudes_list = []
    bbox_coords_list = []
    for topleft_index in topleft_indices:
        x1, x2, y1, y2 = topleft_index[0], topleft_index[0]+512, topleft_index[1], topleft_index[1]+512
        bound = predict_boundaries_512(image[:, :, x1:x2, y1:y2])
        '''
        edge_weights = hg.weight_graph(graph,bound,hg.WeightFunction.mean)
        tree, altitudes = hg.watershed_hierarchy_by_dynamics(graph, edge_weights)
        trees_list.append(tree)
        altitudes_list.append(altitudes)
        bbox_coords_list.append((x1, x2, y1, y2))
        
    for k, T in enumerate(T_values):
        pred_segmentation = np.zeros((2048, 2048))
        for j in range(len(topleft_indices)):
            x1, x2, y1, y2 = bbox_coords_list[j]
            tree = trees_list[j]
            altitudes = altitudes_list[j]
            pred_segmentation_j = hg.labelisation_horizontal_cut_from_threshold(tree,altitudes,threshold=T).astype(float) 
            binary_field_extent_j = find_boundaries(pred_segmentation_j)==0
            pred_segmentation[x1:x2, y1:y2] = binary_field_extent_j
        ndimage.label(pred_segmentation, output=pred_segmentation)
        ndimage.grey_dilation(pred_segmentation, size=(3, 3), output=pred_segmentation)

        # remove small fields 
        unique_labels, counts = np.unique(pred_segmentation, return_counts=True)
        remove_mask = np.isin(pred_segmentation, unique_labels[counts < 300]) 
        pred_segmentation[remove_mask] = 0

        # remove fields that don't intersect with CDL cultivated land
        field_labels = np.unique(CDL*pred_segmentation) 
        remove_mask = ~np.isin(pred_segmentation, field_labels) # remove fields that don't intersect with CDL cultivated land
        pred_segmentation[remove_mask] = 0

        # remove fields at image border
        border_labels = np.unique(image_border*pred_segmentation)
        remove_mask = np.isin(pred_segmentation, border_labels)
        pred_segmentation[remove_mask] = 0

        # remove fields fully contained in top and left image margins (these will be computed in the overlapping neighboring tiles)
        bottom_right_corner_labels = np.unique(image_bottom_right_corner*pred_segmentation)
        remove_mask = ~np.isin(pred_segmentation, bottom_right_corner_labels)
        pred_segmentation[remove_mask] = 0
    
        results = [shape(s) for (s, v) in shapes(source=pred_segmentation, mask=pred_segmentation>0, transform=transform)]
        results_simplified = [polygon.simplify(tolerance=8, preserve_topology=True) for polygon in results]
        df = pd.DataFrame({'geometry':results_simplified})
        df.to_csv(f'testing/outputs-attempt1/{image_number}-{k+1}.csv', index=False)
    '''

[22:30:13] /work/mxnet/src/operator/nn/./cudnn/./cudnn_algoreg-inl.h:96: Running performance tests to find the best convolution algorithm, this can take a while... (set the environment variable MXNET_CUDNN_AUTOTUNE_DEFAULT to 0 to disable)
[22:30:15] /work/mxnet/src/operator/nn/./cudnn/cudnn_pooling-inl.h:374: 0D pooling is not supported by cudnn, MXNet 0D pooling is applied.
[22:30:17] /work/mxnet/src/operator/nn/./cudnn/./cudnn_algoreg-inl.h:96: Running performance tests to find the best convolution algorithm, this can take a while... (set the environment variable MXNET_CUDNN_AUTOTUNE_DEFAULT to 0 to disable)
  9%|▉         | 9/100 [02:12<22:19, 14.71s/it]


KeyboardInterrupt: 